In [1]:
# Import necessary libraries
import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from sqlalchemy import create_engine, text
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import time
from pathlib import Path
from urllib.parse import urljoin
from selenium.webdriver.chrome.options import Options
import pandas as pd
load_dotenv()

True

In [2]:
url = "https://books.toscrape.com/catalogue/page-1.html"

r = requests.get(url)
print(r.status_code)

200


In [3]:
def make_driver():
    chrome_options = Options()

    # "--headless=new" works better with newer Chrome versions
    #chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    # Selenium Manager handles ChromeDriver automatically
    return webdriver.Chrome(options=chrome_options)


driver = make_driver()

try:
    driver.get(url)
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "article.product_pod"))
    )
    page_source = driver.page_source

    print(driver.title)
finally:
    driver.quit()

All products | Books to Scrape - Sandbox


In [4]:
# Lists to store the extracted data
book_names = []
prices = []
in_stocks = []
ratings = []
book_urls = []
book_images = []
categories = []


# Assuming you know the total pages
total_pages = 50  # Adjust this based on the actual number of pages
BOOK_BASE_URL = "https://books.toscrape.com/catalogue/"


detail_driver = make_driver()
try:
    for page_number in range(1, total_pages + 1):
        url = f"https://books.toscrape.com/catalogue/page-{page_number}.html"
        detail_driver.get(url)
        WebDriverWait(detail_driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "article.product_pod"))
        )

        # Parse the catalogue page currently displayed in Chrome.
        soup = BeautifulSoup(detail_driver.page_source, "html.parser")

        # Locate each book card in the catalogue.
        books = soup.select("article.product_pod")

        for book in books:
            # Extract book name
            title_element = book.select_one("h3 a")
            price_element = book.select_one("p.price_color")
            availability_element = book.select_one("p.instock.availability")
            rating_element = book.select_one("p.star-rating")

            book_name = (
                title_element.get("title") if title_element else None
            )

            # Extract prices
            price = (
                price_element.get_text(strip=True)
                .replace("£", "")
                .replace(",", "")
                if price_element
                else None
            )

            # Extract in-stock status
            in_stock = (
                availability_element.get_text(" ", strip=True)
                if availability_element
                else None
            )

            # Extract rating
            rating_classes = (
                rating_element.get("class", []) if rating_element else []
            )
            rating = next(
                (value for value in rating_classes if value != "star-rating"),
                None,
            )

            # Extract book URL
            book_url = (
                urljoin(BOOK_BASE_URL, title_element["href"])
                if title_element and title_element.get("href")
                else None
            )

            # Categories are available on the book detail page breadcrumb.
            category = None
            if book_url:
                detail_driver.get(book_url)
                WebDriverWait(detail_driver, 10).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "ul.breadcrumb"))
                )
                detail_soup = BeautifulSoup(detail_driver.page_source, "html.parser")
                category_element = detail_soup.select_one(
                    "ul.breadcrumb li:nth-of-type(3) a"
                )
                category = (
                    category_element.get_text(strip=True)
                    if category_element
                    else None
                )

            # Extract book image URL
            book_image = (
                urljoin(BOOK_BASE_URL, book.select_one("img")["src"])
                if book.select_one("img")
                else None
            )

            # Append the extracted data to the respective lists
            book_names.append(book_name)
            prices.append(price)
            in_stocks.append(in_stock)
            ratings.append(rating)
            categories.append(category)
            book_urls.append(book_url)
            book_images.append(book_image)
finally:
    detail_driver.quit()

# Create a DataFrame from the extracted data
data = {
    "book_names": book_names,
    "availability": in_stocks,
    "ratings": ratings,
    "prices": prices,
    "categories": categories,
    "book_urls": book_urls,
    "book_images": book_images,
}

books_df = pd.DataFrame(data)

In [5]:
# Save the DataFrame to a CSV file
books_df.to_csv("../data/raw_data/books_data.csv", index=False)

In [6]:
# Convert the cleaned price values to numbers.
books_df["prices"] = pd.to_numeric(books_df["prices"], errors="coerce")

In [7]:
# Convert the rating column to numeric values for easier analysis
rating_mapping = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5,
}
books_df["ratings"] = books_df["ratings"].map(rating_mapping)

In [8]:
# Store all categories discovered in the book breadcrumbs.
books_df["categories"] = books_df["categories"].astype("category")

In [9]:
# Save the cleaned DataFrame to a CSV file
books_df.to_csv("../data/cleaned_data/books_data.csv", index=False)

In [10]:
# Create a database in PostgreSQL and store the data in a table
# Define the database connection parameters
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")

In [11]:
# Create the PostgreSQL database if it does not already exist
admin_engine = create_engine(
    f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/postgres",
    isolation_level="AUTOCOMMIT",
)
with admin_engine.connect() as connection:
    database_exists = connection.execute(
        text("SELECT 1 FROM pg_database WHERE datname = :db_name"),
        {"db_name": db_name},
    ).scalar()
    if not database_exists:
        quoted_db_name = admin_engine.dialect.identifier_preparer.quote_identifier(
            db_name
        )
        connection.exec_driver_sql(f"CREATE DATABASE {quoted_db_name}")

admin_engine.dispose()
engine = create_engine(
    f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
)

In [12]:
# # Create schema and table in the database
with engine.connect() as connection:
    # Create schema if it doesn't exist
    connection.execute(text("CREATE SCHEMA IF NOT EXISTS pagepulse;"))

    # Create table if it doesn't exist
    connection.execute(text("""
        CREATE TABLE IF NOT EXISTS pagepulse.books_data (
            book_names TEXT ,
            availability TEXT,
            ratings TEXT,
            prices FLOAT,
            categories TEXT,
            book_urls TEXT,
            book_images TEXT
        );
    """))

In [13]:
# Load the saved cleaned data into the PostgreSQL database
with engine.begin() as connection:
    connection.execute(text("CREATE SCHEMA IF NOT EXISTS pagepulse"))
    books_df.to_sql(
        "books_data",
        connection,
        schema="pagepulse",
        if_exists="replace",
        index=False,
    )
print("Data loaded into PostgreSQL database successfully.")

Data loaded into PostgreSQL database successfully.
